In [17]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from scipy.stats import randint, uniform

In [18]:
df_combined = pd.read_parquet("data/df_combined.parquet")

In [19]:
train_mask = df_combined.split == 'train'
val_mask = df_combined.split == 'val'
assert set(df_combined.loc[train_mask, 'Place_ID']) & set(df_combined.loc[val_mask, 'Place_ID']) == set()

X_train = df_combined.loc[train_mask].drop(columns=['target', 'split', 'Place_ID', 'Date'])
y_train = df_combined.loc[train_mask, 'target']
groups_train = df_combined.loc[train_mask, 'Place_ID']   # NEU: separat gesichert, bevor Place_ID weg ist

X_val = df_combined.loc[val_mask].drop(columns=['target', 'split', 'Place_ID', 'Date'])
y_val = df_combined.loc[val_mask, 'target']

## Baseline Model - Dummy Model

In [20]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(np.zeros((len(y_train), 1)), y_train)  # Features irrelevant, Dummy ignoriert X komplett
y_pred = dummy.predict(np.zeros((len(y_val), 1)))

baseline_rmse = mean_squared_error(y_val, y_pred, squared=False)
print(f"Baseline RMSE: {baseline_rmse:.2f}")

Baseline RMSE: 42.12


### Sensor Altitude Feature — Reasoning Summary

- Initial finding: sensor_altitude columns ranked among top features by importance, with moderate target correlation (~-0.31)
- Hypothesis raised: could reflect terrain elevation, which plausibly affects pollutant accumulation (inversion layers, valley/basin effects)
- Verification: value range (~828–844 km) matches Sentinel-5P's known orbital altitude (~824 km), not terrain elevation (max ~8.8 km globally) — confirms this is satellite viewing-geometry metadata, not a ground property
- Risk: likely acts as an indirect proxy for geographic position (latitude/orbit geometry) rather than a causal driver of PM2.5 — no physical mechanism linking satellite altitude to ground-level pollution
- Decision: excluded from final feature set to avoid relying on a non-causal, potentially non-generalizable shortcut
- Future work: true terrain elevation (e.g. DEM/SRTM data) could be a valuable feature if location coordinates become available

In [21]:
drop_metadata = ['L3_AER_AI_sensor_altitude', 'L3_NO2_sensor_altitude', 'L3_CO_sensor_altitude']
df_combined = df_combined.drop(columns=drop_metadata)

## Advanced Data Split with GroupKFold

In [22]:
gkf = GroupKFold(n_splits=5)
fold_scores = []

for train_idx, val_idx in gkf.split(X_train, groups=groups_train):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = XGBRegressor(random_state=42)
    model.fit(X_tr, y_tr)
    rmse = mean_squared_error(y_va, model.predict(X_va), squared=False)
    fold_scores.append(rmse)
    print(f"Fold RMSE: {rmse:.3f}")

print(f"\nMean RMSE: {np.mean(fold_scores):.3f} ± {np.std(fold_scores):.3f}")

Fold RMSE: 36.049
Fold RMSE: 34.101
Fold RMSE: 33.053
Fold RMSE: 42.846
Fold RMSE: 31.371

Mean RMSE: 35.484 ± 3.981


In [23]:


param_dist = {
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.29),
    'n_estimators': randint(100, 800),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 10),   # Regularisierung gegen Overfitting auf kleine Location-Gruppen
    'reg_lambda': uniform(0.5, 4.5)      # L2-Regularisierung, hilfreich bei vielen korrelierten Satellite-Features
}

random_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=100,
    cv=GroupKFold(n_splits=5),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

random_search.fit(X_train, y_train, groups=groups_train)
print(f"Best params: {random_search.best_params_}")
print(f"Best CV RMSE (on 80% train): {-random_search.best_score_:.3f}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params: {'colsample_bytree': 0.7390476857930735, 'learning_rate': 0.019223357630697886, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 744, 'reg_lambda': 2.573504456147266, 'subsample': 0.682533487362317}
Best CV RMSE (on 80% train): 33.404


Best params: {
'colsample_bytree': 0.7390476857930735,

'learning_rate': 0.019223357630697886, 

'max_depth': 8, 

'min_child_weight': 1, 

'n_estimators': 744, 

'reg_lambda': 2.573504456147266, 

'subsample': 0.682533487362317}

Best RMSE: 33.038

In [24]:
pred = random_search.predict(X_val)
print(f"Held-out VAL RMSE (20%): {np.sqrt(mean_squared_error(y_val, pred)):.2f}")

Held-out VAL RMSE (20%): 30.09
